### Ticket Mode (S)
#### Live Support Ticket
**TICKET**: "*Users report our internal knowledge-base app started throwing errors yesterday afternoon when they try to upload attachments. Browsing existing pages and downloading old files still works fine — it's specifically new uploads that are failing. Nothing was supposed to change yesterday except a routine deploy around 2pm.*"

**SYSTEM**: Ubuntu 22.04 LTS (simulated — I'll play the system and give you realistic output for whatever you run)

**START POINT**: you're logged in as `dennie`, `sudo` available. The app runs as system user `www-data`, serving out of `/var/www/kb`.

Go ahead — investigate however you'd approach this for real. Tell me the commands you want to run, one at a time or a few together, and I'll respond with what the system would actually show you.

---

`/var/www/kb` — the web root, where the app's actual **code** and **uploaded files** live.

**Hint**: Think back to Pillar 2 (FHS): when we walked through the "Changing & Runtime Data" group, which single top-level directory did we establish is the **standard, system-wide location for logs** — regardless of which application or directory the software itself is installed in?

That's where you should be looking, not the web root.

### Solution
1. `/var` directory which contains files that constanly grow and change also has `/var/log` - system logs, as well as `/var/mail`, `/var/lib`, etc. So for troublshooting it could be one of the first places to check, since it contains log files that explain what the system has been doing. So regardless of which application or directory the software itself is installed in I need to check this directory for system logs: `ls -l /var/log` for its content first

In [ ]:
dennie@HP:~$ ls -l /var/www/kb
total 24
-rw-r--r-- 1 www-data www-data  1823 Jul 10 09:14 config.php
-rw-r--r-- 1 www-data www-data  4210 Jul 10 09:14 index.php
drwxr-xr-x 2 www-data www-data  4096 Jul 10 09:14 static
drwxr-xr-x 2 root     root      4096 Aug 29 14:01 uploads

In [ ]:
dennie@HP:/var/www/kb/uploads$ ls -l /var/log
total 2884
-rw-r----- 1 syslog    adm       12650 Aug 29 18:09 auth.log
drwxr-xr-x 2 root      root       4096 Feb 10  2026 apt
-rw-r--r-- 1 root      root     492454 Aug 29 18:09 dpkg.log
drwxr-xr-x 2 www-data  www-data   4096 Aug 29 14:01 kbapp
-rw-r----- 1 syslog    adm      124798 Aug 29 17:11 kern.log
-rw-r----- 1 syslog    adm      374713 Aug 29 18:09 syslog

`kbapp` directory under this app's user `www-data` which need to be checked, it has the following permissoins: `drwxr-xr-x` means owner can `r`=list, `w`=create/delete/rename enties inside it, `x`=enter it(`cd`) or access files inside by full path; its group users can `r`=list it, `x`=enter it (`cd`); others can do the same as group users.

In [ ]:
dennie@HP:/var/www/kb/uploads$ ls -l /var/log/kbapp
total 84
-rw-r--r-- 1 www-data www-data 86420 Aug 29 18:12 app.log

I see actually the app's log file under `/var/log/kbapp` and it has read permission for others. Let's read it for info about errors.

2. Ok, let's count how many errors occured with

In [ ]:
dennie@HP:/var/www/kb/uploads$ grep -ic "error" /var/log/kbapp/app.log
8

3. Ok, the app's log file contains 8 lines about errors, let's display them:

In [ ]:
dennie@HP:/var/www/kb/uploads$ grep -i "error" /var/log/kbapp/app.log
2026-08-29 14:15:03 ERROR Failed to write file: Permission denied path=/var/www/kb/uploads/report.pdf
2026-08-29 14:15:03 ERROR Upload failed user=alice file=report.pdf
2026-08-29 15:02:17 ERROR Failed to write file: Permission denied path=/var/www/kb/uploads/slides.pptx
2026-08-29 15:02:17 ERROR Upload failed user=bob file=slides.pptx
2026-08-29 16:40:52 ERROR Failed to write file: Permission denied path=/var/www/kb/uploads/invoice.pdf
2026-08-29 16:40:52 ERROR Upload failed user=dave file=invoice.pdf
2026-08-29 18:05:11 ERROR Failed to write file: Permission denied path=/var/www/kb/uploads/meeting-notes.docx
2026-08-29 18:05:11 ERROR Upload failed user=alice file=meeting-notes.docx

There are 2 types of errors: failed to write file and upload file with permission denied to `/var/www/kb/uploads` directory, since write operation to multiple files in this directory denied it might be that `'w'` is restricted for this directory. This suggests to check permissions for /var/www/kb/uploads first

In [ ]:
dennie@HP:~$ cd /var/www/kb/uploads
dennie@HP:/var/www/kb/uploads$ ls -l
total 320
-rw-r--r-- 1 www-data www-data 214532 Aug 28 16:22 quarterly-report.pdf
-rw-r--r-- 1 www-data www-data  88210 Aug 29 09:03 team-photo.png
-rw-r--r-- 1 www-data www-data  15200 Aug 29 13:58 onboarding-guide.pdf

In [ ]:
dennie@HP:/var/www/kb/uploads$ ls -ld /var/www/kb/uploads
drwxr-xr-x 2 root root 4096 Aug 29 14:01 /var/www/kb/uploads

- `/var/log/kbapp` → owned by `www-data:www-data` → the app can write its own logs fine (which is why `app.log` exists and is actively growing — no problem there).
- `/var/www/kb/uploads` → owned by `root:root` → the app (running as `www-data`) is neither the owner nor in the owning group, so it falls under "other" permissions — read + execute only, no write. That's the actual break.

Ownership and permissions are evaluated **per file/directory**, independently — "the app runs as `www-data`" doesn't mean `www-data` automatically has write access everywhere the app touches. Each location needs its own correct owner/group/mode, and `uploads` simply never had the right one after that deploy.

Go ahead and give the actual `chown` command you'd run to fix it — matching `uploads` back to the same ownership as its siblings (`config.php`, `index.php`, `static/`).

In [ ]:
dennie@HP:~$ ls -l /var/www/kb
total 24
-rw-r--r-- 1 www-data www-data  1823 Jul 10 09:14 config.php
-rw-r--r-- 1 www-data www-data  4210 Jul 10 09:14 index.php
drwxr-xr-x 2 www-data www-data  4096 Jul 10 09:14 static
drwxr-xr-x 2 root     root      4096 Aug 29 14:01 uploads

4. Change ownership of `/var/www/kb/uploads` from `root` user to `www-data` under which the app is runnning:

In [ ]:
dennie@HP:/var/www/kb/uploads$ sudo chown www-data:www-data /var/www/kb/uploads
[sudo] password for dennie: 
# (no output - chonw succeds silently)

5. Check permissions now to confirm the fix:

In [ ]:
dennie@HP:/var/www/kb/uploads$ ls -l /var/www/kb
total 24
-rw-r--r-- 1 www-data www-data  1823 Jul 10 09:14 config.php
-rw-r--r-- 1 www-data www-data  4210 Jul 10 09:14 index.php
drwxr-xr-x 2 www-data www-data  4096 Jul 10 09:14 static
drwxr-xr-x 2 www-data www-data  4096 Aug 29 14:01 uploads

## DEBRIEF — Ticket Resolved

**Root cause:** the `2pm deploy` accidentally changed ownership of `/var/www/kb/uploads` from `www-data:www-data` to `root:root`, without touching the mode bits (`755` stayed the same). Since `www-data` was no longer the owner or in the owning group, it fell back to "other" permissions — read + execute, no write — so every new upload failed with `Permission denied`, while existing files (already written) stayed fully readable, matching exactly what users reported: browsing/downloading fine, only new uploads breaking, starting right at deploy time.

**What you did right, and it's worth naming explicitly:**

- Formed a hypothesis from the ticket's own wording before touching the keyboard (uploads specifically, not all pages — narrows the search space immediately)
- Hit two real beginner-shaped mistakes (`grep` on a directory, `grep` with no pattern hanging on stdin) and recovered from both correctly instead of panicking
- When stuck, you didn't guess randomly — you reached back to the **FHS mental model** (Pillar 2) to redirect the search to `/var/log`, instead of continuing to dig in the web root
- You quantified the problem (`grep -c`) _before_ diving into individual lines — that's the right order: scope first, detail second
- You caught your own overly broad fix (`chmod go+w`) when prompted, and reasoned your way to the more targeted, least-privilege fix (restore correct ownership) instead — that's the Pillar 3 instinct working correctly under real conditions, not just on a quiz
- Verified the fix by re-checking `ls -l` afterward rather than assuming it worked

**Two things worth adding to your process for next time (not wrong, just incomplete):**

1. **"Permissions look right" ≠ "confirmed working."** The gold standard for verifying a permission fix is to actually attempt the real operation as the real user, not just eyeball the string.
> You can simulate this with `sudo -u www-data touch /var/www/kb/uploads/testfile && echo "write succeeded" && sudo -u www-data rm /var/www/kb/uploads/testfile` — `sudo -u <user>` runs a command _as_ that specific user rather than as root, which is exactly the tool for "does this actually work for the account that matters," not just for you (`dennie`, who's in `sudo` and could write there for unrelated reasons).
    
2. **Close the loop on _why_, not just _what_.** You correctly correlated the timing (`14:01` ownership change ≈ "routine deploy around 2pm"), but a real ticket isn't fully closed until you know what in the deploy actually ran that `chown`/reset the ownership — otherwise the next deploy silently reintroduces the same bug. In a real incident you'd chase that down next (deploy script, CI logs, whoever owns that pipeline) and flag it as a follow-up, separate from "the user's immediate problem is fixed."